# Módulo 0 — Fundamentos: de la probabilidad a las matrices de densidad

**Objetivo de este módulo:** entender qué es una *matriz de densidad de kernel* (KDM) desde cero — la idea físico-matemática que toma prestada de mecánica cuántica, y su representación exacta como tensor en el código de `kdm-torch`. Todo lo que sigue (clasificación, regresión, estimación de densidad, generación) es una aplicación de esta única idea.

**Fuente matemática:** González, Ramos-Pollán & Gallego (2025), *Quantum Machine Intelligence* 7:94 — Sección 3 (`docs/s42484-025-00299-9.pdf`).
**Fuente de código:** `external/kdm/kdm/utils.py`.

No hace falta saber mecánica cuántica — solo se usa una pieza de su formalismo (la matriz de densidad) como una forma conveniente de representar mezclas de puntos con pesos, que resulta tener propiedades muy útiles para estimar densidades de probabilidad.

## 1. Estados puros y mixtos (§3.1)

En mecánica cuántica, un **estado puro** se describe con una función de onda $|\psi\rangle$ que vive en un espacio de Hilbert $H$. La regla de Born dice que la probabilidad de "encontrar" el sistema en otro estado $|\phi\rangle$ es el cuadrado de la proyección:

$$p(\phi\,|\,\psi) = |\langle\phi|\psi\rangle|^2 \tag{análogo a Eq. 2, caso puro}$$

Cuando además hay incertidumbre **clásica** sobre cuál estado puro tiene el sistema — una mezcla estadística de $N$ estados $|\psi_i\rangle$, cada uno con probabilidad $p_i$ ($\sum_i p_i = 1$) — se dice que el sistema está en un **estado mixto**, representado por la **matriz de densidad**:

$$\rho = \sum_i p_i\, |\psi_i\rangle\langle\psi_i| \tag{Eq. 1}$$

La regla de Born se extiende para calcular la probabilidad de encontrar un sistema en estado $\rho$ en el estado $|\phi\rangle$:

$$p(\phi\,|\,\rho) = \mathrm{tr}(|\phi\rangle\langle\phi|\,\rho) = \sum_i p_i\, |\langle\phi|\psi_i\rangle|^2 \tag{Eq. 2}$$

**La idea clave del paper**: en vez de usar esto para física cuántica, se usa $\rho$ como una estructura de datos para representar *distribuciones de probabilidad clásicas* — cada $|\psi_i\rangle$ es un punto de datos (o su imagen bajo un kernel), y $p_i$ es su peso en la mezcla.

## 2. La KDM: matriz de densidad en el RKHS de un kernel (§3.2)

Una **KDM** es una matriz de densidad definida en el *Reproducing Kernel Hilbert Space* (RKHS) inducido por un kernel $k$. Formalmente, se define como una tripleta:

$$\rho = (C, p, k), \qquad C = (x^{(1)}, \ldots, x^{(m)}) \in X^m,\quad p = (p_1,\ldots,p_m) \in \mathbb{R}^m$$

donde $k: X \times X \to \mathbb{R}$ satisface $k(x,x)=1$ para todo $x$, y $p_i \ge 0$, $\sum_i p_i = 1$.

- $C$ = los **componentes** de la KDM (los puntos de datos).
- $p$ = los **pesos de mezcla** (probabilidad de cada componente).
- $k$ = el kernel — **no se guarda en el tensor**, es un objeto aparte (`RBFKernelLayer`, `CosineKernelLayer`, ...) que se aplica cuando se necesita evaluar la KDM.

La **función de proyección** asociada a una KDM $\rho$ es:

$$f_\rho(x) = \sum_{x^{(i)} \in C} p_i\, k^2(x, x^{(i)}) \tag{Eq. 3}$$

que, multiplicada por una constante de normalización $M_k$ que depende del kernel, se convierte en una función de densidad (caso continuo) o de masa (caso discreto) de probabilidad válida:

$$\hat{f}_\rho(x) = M_k\, f_\rho(x) \tag{Eq. 4}$$

## 3. KDM discreta: el kernel coseno (§3.2.1)

Para representar distribuciones categóricas, el paper usa $X = \mathbb{R}^n$ con el **kernel coseno**:

$$k_{\cos}(x,y) = \frac{\langle x, y \rangle}{\|x\|\,\|y\|}$$

cuya constante de normalización es $M_{k_{\cos}} = 1$. Si $X = \{e^{(1)}, \ldots, e^{(n)}\}$ es la base canónica de $\mathbb{R}^n$, la Ec. 4 define una PMF categórica válida sobre esa base:

$$\hat{f}_\rho(e^{(i)}) = f_\rho(e^{(i)}) \tag{Eq. 5}$$

**Ejemplo del paper** (lo reproducimos en código abajo): la distribución categórica $p=(0.2, 0.3, 0.5)$ se puede representar de **dos formas distintas** con una KDM:

- $\rho_0$ = una mezcla genuina de 3 componentes (la base canónica $e^{(1)}, e^{(2)}, e^{(3)}$), con pesos $(0.2, 0.3, 0.5)$.
- $\rho_1$ = un **único** componente puro $(\sqrt{0.2}, \sqrt{0.3}, \sqrt{0.5})$, con peso $1$.

Esto funciona porque, por la regla de Born (Ec. 2), $|\langle e^{(i)} | \psi \rangle|^2 = \psi_i^2$ — si $\psi = (\sqrt{p_1},\ldots,\sqrt{p_n})$, entonces $\psi_i^2 = p_i$ exactamente. Verificamos esto numéricamente.

In [1]:
import torch
from kdm.utils import dm2comp, comp2dm, samples2dm, pure2dm, dm2discrete

torch.manual_seed(42)

# --- rho_0: mezcla genuina de 3 componentes (base canonica), pesos (0.2, 0.3, 0.5) ---
p = torch.tensor([0.2, 0.3, 0.5])
C0 = torch.eye(3)                     # e(1), e(2), e(3) como filas
dm_rho0 = comp2dm(p.unsqueeze(0), C0.unsqueeze(0))   # (bs=1, n=3, d+1=4)
print("rho_0 (mezcla de 3):", dm_rho0.shape)

# --- rho_1: un solo componente puro, psi = sqrt(p) ---
psi = torch.sqrt(p)
dm_rho1 = pure2dm(psi.unsqueeze(0))   # (bs=1, n=1, d+1=4)
print("rho_1 (estado puro):", dm_rho1.shape)

# --- Ambas deben dar la MISMA distribucion categorica via dm2discrete (Eq. 5) ---
print("\ndm2discrete(rho_0) =", dm2discrete(dm_rho0))
print("dm2discrete(rho_1) =", dm2discrete(dm_rho1))
print("p original          =", p)

rho_0 (mezcla de 3): torch.Size([1, 3, 4])
rho_1 (estado puro): torch.Size([1, 1, 4])

dm2discrete(rho_0) = tensor([[0.2000, 0.3000, 0.5000]])
dm2discrete(rho_1) = tensor([[0.2000, 0.3000, 0.5000]])
p original          = tensor([0.2000, 0.3000, 0.5000])


Las tres filas deben coincidir (salvo error de punto flotante) — dos representaciones tensoriales completamente distintas (`n=3` vs. `n=1` componentes) codifican **exactamente la misma distribución de probabilidad**. Esta flexibilidad — poder representar la misma distribución como una mezcla explícita o como un único punto — es central en cómo `KDMLayer` combina información más adelante.

## 4. KDM continua: el kernel RBF (§3.2.2)

Para variables continuas, el paper usa un kernel radial $k(x,y) = K(x-y)$ que satisface ciertas condiciones (Ec. 6) — el ejemplo usado en este trabajo es el **kernel Gaussiano (RBF)**:

$$k_{\mathrm{rbf},\sigma}(x,y) = e^{-\frac{\|x-y\|^2}{2\sigma^2}}$$

Con este kernel, la Ec. 4 corresponde exactamente a un estimador de densidad por kernels (KDE) con kernel Gaussiano y ancho de banda $\sigma$. El **Teorema 2 (Parzen, 1962)** dice que si $D = \{x^{(1)},\ldots,x^{(\ell)}\}$ son muestras iid de una densidad $f$, y se construye la KDM $\rho_x = (D,\, p=(1/\ell,\ldots,1/\ell),\, k_{\mathrm{rbf},\sigma})$ — es decir, **cada muestra es un componente con peso uniforme** — entonces $\hat{f}_{\rho_x}(x) \to f(x)$ en probabilidad cuando $\ell\to\infty$ y $\sigma\to 0$ apropiadamente.

Esto es exactamente lo que hace `samples2dm`: construye una KDM empírica a partir de un lote de muestras, dándole a cada una el mismo peso.

## 5. La representación tensorial exacta en el código

El código nunca guarda la tripleta $(C, p, k)$ como tal — usa **un solo tensor** de forma `(bs, n, d+1)`: `bs` = tamaño de lote, `n` = número de componentes, `d` = dimensión de cada componente. La columna 0 son los pesos $p$; las columnas $1{:}$ son los componentes $C$.

```python
# external/kdm/kdm/utils.py:6-15
def dm2comp(dm):
    return dm[:, :, 0], dm[:, :, 1:]   # w: (bs,n) = p ;  v: (bs,n,d) = C

# external/kdm/kdm/utils.py:18-27
def comp2dm(w, v):
    return torch.cat((w.unsqueeze(-1), v), dim=2)
```

Verificamos el *roundtrip* (`dm → comp → dm` no debe perder información):

In [2]:
w, v = dm2comp(dm_rho0)
print("w (pesos, = p):", w)
print("v (componentes, = C):\n", v)

dm_reconstruida = comp2dm(w, v)
assert torch.allclose(dm_reconstruida, dm_rho0)
print("\nRoundtrip dm -> (w,v) -> dm: OK, tensores identicos")

w (pesos, = p): tensor([[0.2000, 0.3000, 0.5000]])
v (componentes, = C):
 tensor([[[1., 0., 0.],
         [0., 1., 0.],
         [0., 0., 1.]]])

Roundtrip dm -> (w,v) -> dm: OK, tensores identicos


## 6. `pure2dm` — estados puros como caso especial ($n=1$)

```python
# external/kdm/kdm/utils.py:45-55
def pure2dm(psi):
    ones = torch.ones_like(psi[:, 0:1])
    dm = torch.cat((ones.unsqueeze(1), psi.unsqueeze(1)), dim=2)
    return dm   # (bs, 1, d+1) -- un unico componente, peso = 1
```

Ya lo usamos arriba para construir $\rho_1$. Es literalmente la Ec. 1 con $N=1$: $\rho = 1\cdot|\psi\rangle\langle\psi|$ — un estado puro, sin incertidumbre clásica adicional.

## 7. `samples2dm` — KDM empírica desde datos crudos

```python
# external/kdm/kdm/utils.py:30-42
def samples2dm(samples):
    nonzero = (samples != 0).any(dim=-1).to(samples.dtype)
    w = nonzero / nonzero.sum(dim=-1, keepdim=True)   # peso uniforme 1/l
    return comp2dm(w, samples)
```

Cada muestra se vuelve un componente con peso $1/\ell$ — exactamente el $\rho_x$ del Teorema de Parzen (§4 arriba). Las filas que son todo-ceros se ignoran (peso 0) — un truco práctico para poder usar tensores de tamaño fijo con lotes de tamaño variable (relleno con ceros).

## 8. Ejercicio práctico con MNIST

Usamos MNIST (ya descargado localmente) para dos ejercicios distintos, cada uno alineado con la interpretación correcta de cada función:

- **KDM discreta** (`samples2dm` + `dm2discrete`) sobre las **etiquetas** de dígito (vectores one-hot de 10 dimensiones) — esto es exactamente la misma convención que usa NPC para sus atributos (ver `npc-models/dataset.py`), así que este ejercicio conecta directo con el resto de la tesis.
- **Estado puro** (`pure2dm`) sobre una **imagen** aplanada y normalizada — aquí NO interpretamos `dm2discrete` sobre los 784 píxeles (no es una distribución categórica significativa sobre píxeles); solo verificamos la mecánica del tensor. La interpretación continua real llega en el Módulo 1 con `KDMLayer` + kernel RBF.

In [3]:
import torchvision

mnist = torchvision.datasets.MNIST(root="../../../data/mnist", train=True, download=False)

# --- Ejercicio A: KDM discreta sobre las ETIQUETAS (10 clases) ---
N = 500
labels = mnist.targets[:N]                                  # (N,) enteros 0-9
labels_onehot = torch.nn.functional.one_hot(labels, num_classes=10).float()  # (N, 10)

dm_labels = samples2dm(labels_onehot.unsqueeze(0))          # (bs=1, n=N, d=10+1)
print("KDM de etiquetas:", dm_labels.shape, "-> n =", dm_labels.shape[1], "componentes (uno por muestra)")

distribucion_recuperada = dm2discrete(dm_labels).squeeze(0)
distribucion_real = torch.bincount(labels, minlength=10).float() / N

print("\ndistribucion recuperada via dm2discrete:", distribucion_recuperada.numpy().round(3))
print("distribucion real (conteo directo):      ", distribucion_real.numpy().round(3))
assert torch.allclose(distribucion_recuperada, distribucion_real, atol=1e-5)
print("\nOK: dm2discrete recupera exactamente la frecuencia empirica de cada digito.")

KDM de etiquetas: torch.Size([1, 500, 11]) -> n = 500 componentes (uno por muestra)

distribucion recuperada via dm2discrete: [0.1   0.132 0.104 0.1   0.104 0.078 0.09  0.104 0.078 0.11 ]
distribucion real (conteo directo):       [0.1   0.132 0.104 0.1   0.104 0.078 0.09  0.104 0.078 0.11 ]

OK: dm2discrete recupera exactamente la frecuencia empirica de cada digito.


In [4]:
# --- Ejercicio B: estado puro desde UNA imagen (mecanica del tensor, no interpretacion categorica) ---
import numpy as np

imagen, etiqueta = mnist[0]
x = torch.tensor(np.array(imagen), dtype=torch.float32).flatten()   # (784,)
x_normalizada = x / x.norm()                                        # ||x|| = 1, requerido por el kernel coseno

dm_imagen = pure2dm(x_normalizada.unsqueeze(0))   # (bs=1, n=1, d=784+1)
print(f"Digito real: {etiqueta}")
print("KDM de la imagen:", dm_imagen.shape, "-> n=1 (estado puro), d=784 (pixeles)")

w, v = dm2comp(dm_imagen)
print("peso w =", w.item(), "(debe ser 1.0, un unico componente)")
print("||v|| =", v.norm().item(), "(debe ser 1.0, normalizamos antes)")

Digito real: 5
KDM de la imagen: torch.Size([1, 1, 785]) -> n=1 (estado puro), d=784 (pixeles)
peso w = 1.0 (debe ser 1.0, un unico componente)
||v|| = 0.9999998807907104 (debe ser 1.0, normalizamos antes)


## Resumen y próximo módulo

- Una **KDM** es una matriz de densidad $\rho = \sum_i p_i |\psi_i\rangle\langle\psi_i|$ definida en el RKHS de un kernel — formalmente la tripleta $(C, p, k)$.
- En código, $(C,p)$ viven juntos en **un solo tensor** `(bs, n, d+1)`; el kernel $k$ es un objeto aparte.
- `dm2comp`/`comp2dm` van y vuelven entre el tensor y $(w{=}p,\ v{=}C)$; `pure2dm` construye un estado puro ($n{=}1$); `samples2dm` construye la KDM empírica de Parzen (cada muestra, peso uniforme); `dm2discrete` evalúa la Ec. 5 (PMF categórica vía kernel coseno).
- La misma distribución admite representaciones tensoriales muy distintas (mezcla explícita vs. estado puro) — verificado numéricamente arriba.

**Módulo 1** (`01_kdm_layer_y_clasificacion.ipynb`, pendiente): cómo `KDMLayer` usa esta representación para *aprender* una KDM conjunta y hacer clasificación — el kernel RBF, `init_kdm_layer`, y la mezcla estilo Nadaraya-Watson que calcula `KDMLayer.forward`.